# 神经网络原理精讲 · 训练册

配套《神经网络原理精讲：结构 · 激活 · 损失》使用。

本书有 **27 个动手练习 + 12 道选择题**。每个练习都提供：

- **要做什么**：明确的目标；
- **提示**：卡住时看这里，不是直接给答案；
- **自测**：运行后打印 `[PASS]` / `[FAIL]`，失败会指出哪一项不对。

所有题目只依赖 PyTorch，CPU 即可完成，单题通常 5–15 分钟。参考答案在最后一节。


## 怎么用这本训练册

1. **先读教材对应章节**，再做题。题目与教材章节一一对应，见下方目录。
2. **自己写**：把 `TODO` 替换成代码，不要直接翻答案。
3. **跑自测**：`[FAIL]` 的原因通常是这三类——
   - 变量名写错或没运行上一个代码块；
   - 形状 / dtype 不对；
   - 公式实现有细节错误（比如忘了减均值、忘了 mask）。
4. **看分数**：每部分末尾可以运行 `scoreboard()` 查看累计通过数。
5. **失败是正常的**：训练册的目的就是暴露理解中的小裂缝。

> 提醒：所有实验都固定了随机种子，但不同 PyTorch 版本的初始化细节可能略有差异；自测阈值都留了余量，正常不会误判。


## 目录与对应关系

| 部分 | 对应教材 | 内容 | 题数 |
| --- | --- | --- | --- |
| 一 | 第 1–2 章 | 基础与激活函数 | 6 |
| 二 | 第 3 章 | 损失函数 | 6 |
| 三 | 第 4 章 | 卷积网络 | 5 |
| 四 | 第 6–7 章 | 序列与注意力 | 6 |
| 五 | 第 8–9 章 | 生成模型与图神经网络 | 4 |
| 六 | 全书 | 自测选择题 | 12 |

---


In [ ]:
# 环境准备 + 自测框架（直接运行，不需要修改）
import math
import random

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
random.seed(0)

_results = []


def check(name, fn):
    # fn 返回 True 视为通过；抛异常或返回 False 视为失败
    try:
        ok = bool(fn())
        err = None
    except Exception as e:
        ok = False
        err = e
    _results.append((name, ok))
    if ok:
        print(f"[PASS] {name}")
    else:
        detail = f"  ← {type(err).__name__}: {err}" if err else "  ← 返回 False"
        print(f"[FAIL] {name}{detail}")


def close(a, b, tol=1e-5):
    return torch.allclose(
        torch.as_tensor(a).float(), torch.as_tensor(b).float(), atol=tol, rtol=tol
    )


def quiz(name, given, answer):
    check(name, lambda: str(given).strip().upper() == answer.upper())


def n_params(m):
    return sum(p.numel() for p in m.parameters())


def scoreboard():
    passed = sum(ok for _, ok in _results)
    print(f"\n通过 {passed}/{len(_results)}")


print("PyTorch", torch.__version__, "|", "CUDA" if torch.cuda.is_available() else "CPU")


## 第一部分　基础与激活函数

### 练习 1.1　没有激活的两层网络

**要做什么**

1. 用 `l1`、`l2` 连续处理 `x`，中间**不加任何激活**，得到 `y_two`。
2. 用矩阵乘法手算复合权重 `W = l2.weight @ l1.weight` 与复合偏置 `b = l2.weight @ l1.bias + l2.bias`。
3. 用单层公式 `x @ W.T + b` 得到 `y_one`，验证两者相等。

**提示**：这是「多层线性复合仍是线性」的直接证据，也是为什么必须有激活函数。


In [ ]:
torch.manual_seed(0)
x = torch.randn(4, 3)
l1 = nn.Linear(3, 5)
l2 = nn.Linear(5, 2)

y_two = None   # TODO：连续通过 l1、l2，不加激活
W = None       # TODO：复合权重
b = None       # TODO：复合偏置
y_one = None   # TODO：等价的单层输出


In [ ]:
check("1.1 无激活的两层 == 一层线性", lambda: close(y_two, y_one))


### 练习 1.2　手写 Sigmoid 与它的导数上界

**要做什么**

1. 实现 `sigmoid_t(x)`，不能调用 `torch.sigmoid`。
2. 令 `max_deriv` 为 sigmoid 导数的最大值（可以解析推导，也可以数值扫描）。

**提示**：$\sigma'(z)=\sigma(z)(1-\sigma(z))$，在 $z=0$ 处取最大值。


In [ ]:
def sigmoid_t(x):
    return None   # TODO

xs = torch.linspace(-10, 10, 401)
max_deriv = None  # TODO：填 0.25 或数值计算出的最大值


In [ ]:
check("1.2a sigmoid_t 与 torch.sigmoid 一致",
      lambda: close(sigmoid_t(xs), torch.sigmoid(xs)))
check("1.2b 最大导数为 0.25",
      lambda: abs(max_deriv - 0.25) < 1e-3)


### 练习 1.3　GELU 的精确式与 tanh 近似

**要做什么**

1. 用 `torch.erf` 实现精确 GELU：$x\Phi(x)=x\cdot\frac12[1+\mathrm{erf}(x/\sqrt2)]$。
2. 实现 tanh 近似：$0.5x[1+\tanh(\sqrt{2/\pi}(x+0.044715x^3))]$。

**提示**：`torch.erf` 与 `math.sqrt` 都在手边；近似误差应在 $10^{-3}$ 以下。


In [ ]:
def gelu_exact(x):
    return None   # TODO：用 torch.erf

def gelu_tanh(x):
    return None   # TODO：tanh 近似

xs = torch.linspace(-5, 5, 1001)


In [ ]:
check("1.3a 精确 GELU 与 F.gelu 一致", lambda: close(gelu_exact(xs), F.gelu(xs), 1e-6))
check("1.3b tanh 近似最大误差 < 1e-3",
      lambda: (gelu_exact(xs) - gelu_tanh(xs)).abs().max().item() < 1e-3)


### 练习 1.4　SiLU 与 SwiGLU 的参数量

**要做什么**

1. 实现 `silu_t(x) = x * sigmoid(x)`。
2. 计算 SwiGLU FFN 的参数量（忽略偏置）：$d=4096$，$d_{ff}=11008$，共 3 个矩阵。

**提示**：SwiGLU 是 $W_{down}(\mathrm{SiLU}(W_{gate}x)\odot W_{up}x)$。


In [ ]:
def silu_t(x):
    return None   # TODO

d, d_ff = 4096, 11008
swiglu_params = None   # TODO：3 * d * d_ff


In [ ]:
check("1.4a SiLU 与 F.silu 一致",
      lambda: close(silu_t(torch.linspace(-6, 6, 101)), F.silu(torch.linspace(-6, 6, 101))))
check("1.4b SwiGLU 参数量 = 3*d*d_ff",
      lambda: swiglu_params == 3 * d * d_ff)


### 练习 1.5　初始化：He 与 PyTorch 默认的差异

**要做什么**

实现 `std_ratio(init_fn, depth, width)`：构建 `depth` 层 ReLU MLP（每层 `nn.Linear(width, width)`，用 `init_fn` 初始化权重，激活用 ReLU），返回**最后一层输出标准差 / 第一层输出标准差**。

**提示**：用 `h = F.relu(lin(h))` 逐层前推，把每层的 `h.std()` 记下来。He 初始化应当接近 1，默认初始化会快速衰减。


In [ ]:
def std_ratio(init_fn, depth=30, width=256):
    return None   # TODO

he_init = lambda lin: nn.init.kaiming_normal_(lin.weight, nonlinearity="relu")
default_init = lambda lin: nn.init.kaiming_uniform_(lin.weight, a=math.sqrt(5))


In [ ]:
check("1.5a He 初始化保持方差（比值 > 0.5）",
      lambda: std_ratio(he_init) > 0.5)
check("1.5b PyTorch 默认初始化在深层 ReLU 栈中衰减（比值 < 0.2）",
      lambda: std_ratio(default_init) < 0.2)


### 练习 1.6　深层梯度流

**要做什么**

实现 `grad_norm(depth, act, residual=False)`：

1. 构建 `depth` 个 `nn.Linear(64, 64)`，全部用 `xavier_uniform_` 初始化；
2. 逐层前推：`z = act(lin(h))`；若 `residual=True`，则 `h = h + z`，否则 `h = z`；
3. 对 `h.pow(2).mean()` 反向传播，返回输入梯度的范数。

**提示**：这是教材 1.3 节实验的核心。sigmoid 在 20 层时应当小到 $10^{-6}$ 以下；残差让梯度保持非零（没有归一化时量级甚至可能变大，这正是 ResNet 要配 BatchNorm 的原因）。


In [ ]:
def grad_norm(depth, act, residual=False):
    return None   # TODO

g_sigmoid_20 = None    # TODO：grad_norm(20, torch.sigmoid)
g_relu_20 = None       # TODO：grad_norm(20, F.relu)
g_relu_res_20 = None   # TODO：grad_norm(20, F.relu, residual=True)


In [ ]:
check("1.6a 20 层 sigmoid 梯度几乎消失（< 1e-6）",
      lambda: g_sigmoid_20 < 1e-6)
check("1.6b 20 层 ReLU 梯度比 sigmoid 大至少 4 个数量级",
      lambda: g_relu_20 > g_sigmoid_20 * 1e4)
check("1.6c 残差通路让梯度保持非零",
      lambda: g_relu_res_20 > 1e-2)


In [ ]:
scoreboard()


**第一部分小结**：没有激活的深度是假的；sigmoid 的 $0.25^L$ 会让深层梯度归零；He 初始化与残差是让深网络可训练的两个基本手段。


## 第二部分　损失函数

### 练习 2.1　从 logits 手写交叉熵（数值稳定版）

**要做什么**

实现 `ce_from_logits(logits, target)`，输入形状 `(N, K)`，目标形状 `(N,)`：

1. 用 **log-sum-exp** 技巧计算 $\log p_k=z_k-\log\sum_j e^{z_j}$；
2. 返回批次平均的交叉熵。

**提示**：直接 `logits.softmax(-1).log()` 在 logits 很大时会得到 `-inf`。先用 `max` 做平移再取指数。


In [ ]:
def ce_from_logits(logits, target):
    return None   # TODO

torch.manual_seed(0)
logits_small = torch.randn(8, 5)
target_small = torch.randint(0, 5, (8,))
logits_big = torch.tensor([[1000.0, 0.0]])
target_big = torch.tensor([0])


In [ ]:
check("2.1a 手写 CE 与 F.cross_entropy 一致",
      lambda: close(ce_from_logits(logits_small, target_small),
                    F.cross_entropy(logits_small, target_small)))
check("2.1b 大 logits 下结果有限且正确（接近 0）",
      lambda: torch.isfinite(ce_from_logits(logits_big, target_big))
              and ce_from_logits(logits_big, target_big).item() < 1e-3)


### 练习 2.2　标签平滑与软目标的熵

**要做什么**

$K=3$、真实类别 $y=0$、$\varepsilon=0.1$：

1. 构造平滑后的软目标 `soft_target`；
2. 计算软目标自身的熵 `min_ce`（模型完全拟合软目标时交叉熵的下界）。

**提示**：$y_k'=(1-\varepsilon)y_k+\varepsilon/K$；熵是 $-\sum p\log p$。


In [ ]:
K, eps = 3, 0.1
y = 0
soft_target = None   # TODO：形状 (3,)
min_ce = None        # TODO：软目标的熵


In [ ]:
check("2.2a 目标类别为 0.9333，其余为 0.0333",
      lambda: close(soft_target, torch.tensor([0.9333, 0.0333, 0.0333]), 1e-4))
check("2.2b 熵计算正确",
      lambda: abs(min_ce - (-(soft_target * soft_target.log()).sum()).item()) < 1e-6)


### 练习 2.3　Focal Loss

**要做什么**

实现 `focal_loss(pt, gamma=2.0, alpha=1.0)`，其中 `pt` 是正确类别的概率：

$$\mathrm{FL}(p_t)=-\alpha(1-p_t)^\gamma\log p_t$$

**提示**：$\gamma=0$ 时应退化为交叉熵；$\gamma=2$ 时简单样本（$p_t$ 接近 1）的损失会被大幅压低。


In [ ]:
def focal_loss(pt, gamma=2.0, alpha=1.0):
    return None   # TODO

pt = torch.tensor([0.95, 0.80, 0.55, 0.20])
fl_g0 = None   # TODO：gamma=0
fl_g2 = None   # TODO：gamma=2


In [ ]:
check("2.3a gamma=0 时 Focal == CE",
      lambda: close(fl_g0, -pt.log()))
check("2.3b gamma=2 把 p=0.95 的样本损失压到 CE 的 1% 以下",
      lambda: fl_g2[0].item() < (-pt.log())[0].item() * 0.01)


### 练习 2.4　Dice 与 IoU

**要做什么**

给定二值预测与目标（0/1 浮点张量）：

1. 计算 Dice：$2|A\cap B|/(|A|+|B|)$；
2. 计算 IoU：$|A\cap B|/|A\cup B|$；
3. 验证关系式 $\mathrm{Dice}=2\mathrm{IoU}/(1+\mathrm{IoU})$。

**提示**：交集直接相乘再求和；并集用 `((a + b) > 0).sum()`。


In [ ]:
pred = torch.tensor([1., 1., 1., 1., 0., 0., 0., 0.])
target = torch.tensor([1., 1., 0., 0., 0., 0., 0., 0.])

dice = None
iou = None


In [ ]:
check("2.4a Dice = 2/3", lambda: abs(dice - 2 / 3) < 1e-6)
check("2.4b IoU = 0.5", lambda: abs(iou - 0.5) < 1e-6)
check("2.4c Dice = 2*IoU/(1+IoU)",
      lambda: abs(dice - 2 * iou / (1 + iou)) < 1e-6)


### 练习 2.5　InfoNCE 的温度与熵

**要做什么**

给定 4×4 的相似度矩阵 `sim`：

1. 计算 $\tau=1.0$ 时每行 softmax 的平均熵；
2. 计算 $\tau=0.07$ 时的平均熵；
3. 验证：温度越低，分布越尖锐、熵越小。

**提示**：熵为 $-\sum p\log p$，注意 `p` 中有 0 时用 `p.clamp_min(1e-12)`。


In [ ]:
torch.manual_seed(0)
sim = torch.randn(4, 4) * 2

ent_tau_1 = None      # TODO：tau = 1.0
ent_tau_small = None  # TODO：tau = 0.07


In [ ]:
check("2.5a 低温度让熵变小",
      lambda: ent_tau_small < ent_tau_1 - 0.5)
check("2.5b 熵都非负",
      lambda: ent_tau_1 >= 0 and ent_tau_small >= 0)


### 练习 2.6　类别重加权

**要做什么**

10 个样本里有 9 个多数类（预测正确），1 个少数类（预测错误）。分别计算：

1. 普通交叉熵 `loss_plain`；
2. 带权交叉熵 `loss_weighted`，权重 `w = [1.0, 10.0]`。

**提示**：`F.cross_entropy` 支持 `weight=` 参数。注意 PyTorch 的加权平均是**除以权重之和**，而不是样本数——所以加权后 loss 的绝对数值会变大。


In [ ]:
logits = torch.tensor([[3.0, -3.0]] * 10)
target = torch.tensor([0] * 9 + [1])
w = torch.tensor([1.0, 10.0])

loss_plain = None
loss_weighted = None


In [ ]:
check("2.6a 普通 CE 正确",
      lambda: close(loss_plain, F.cross_entropy(logits, target)))
check("2.6b 加权 CE 正确",
      lambda: close(loss_weighted, F.cross_entropy(logits, target, weight=w)))
check("2.6c 加权后损失变大",
      lambda: loss_weighted.item() > loss_plain.item() * 2)


In [ ]:
scoreboard()


**第二部分小结**：交叉熵传 logits、用 log-sum-exp；软目标改变了损失下界；Focal 压简单样本；Dice 对前景比例不敏感；温度控制对比学习的注意力；重加权改变的是「谁说了算」。


## 第三部分　卷积网络

### 练习 3.1　卷积输出尺寸公式

**要做什么**

实现 `conv_out(size, k, s=1, p=0, d=1)`：

$$H_{out}=\left\lfloor\frac{H+2p-d(k-1)-1}{s}\right\rfloor+1$$

**提示**：`padding` 不改变公式结构；空洞卷积把有效核大小放大为 $d(k-1)+1$。


In [ ]:
def conv_out(size, k, s=1, p=0, d=1):
    return None   # TODO

out_a = None   # TODO：(32, k=3, s=1, p=1) 应保持不变
out_b = None   # TODO：(224, k=7, s=2, p=3) -> 112
out_c = None   # TODO：(32, k=3, s=2, p=1) -> 16
out_d = None   # TODO：(32, k=3, s=1, p=1, d=2) -> 30


In [ ]:
check("3.1a same padding 保持尺寸", lambda: out_a == 32)
check("3.1b 224 -> 112", lambda: out_b == 112)
check("3.1c stride=2 下采样", lambda: out_c == 16)
check("3.1d 空洞卷积输出 30", lambda: out_d == 30)


### 练习 3.2　深度可分离卷积的参数与压缩比

**要做什么**

输入通道 128、输出通道 256、核大小 3（忽略偏置），计算：

1. 标准卷积参数量 `std_params`；
2. 深度可分离卷积参数量 `dw_params`（depthwise + pointwise）；
3. 比值 `ratio = dw_params / std_params`。

**提示**：理论压缩比是 $1/C_{out}+1/k^2$。


In [ ]:
Cin, Cout, kk = 128, 256, 3

std_params = None
dw_params = None
ratio = None


In [ ]:
check("3.2a 标准卷积参数量", lambda: std_params == kk * kk * Cin * Cout)
check("3.2b 深度可分离参数量", lambda: dw_params == kk * kk * Cin + Cin * Cout)
check("3.2c 压缩比符合公式",
      lambda: abs(ratio - (1 / Cout + 1 / kk ** 2)) < 1e-9)


### 练习 3.3　写一个 ResNet BasicBlock

**要做什么**

实现：

```text
conv3x3(c, c, bias=False) -> BN -> ReLU -> conv3x3 -> BN -> add(x) -> ReLU
```

输入输出形状都是 `(N, c, H, W)`。

**提示**：残差相加发生在第二个 BN **之后**、ReLU 之前。


In [ ]:
class BasicBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        # TODO：定义 conv1、bn1、conv2、bn2

    def forward(self, x):
        # TODO
        return None


In [ ]:
check("3.3a 输出形状与输入一致",
      lambda: BasicBlock(64)(torch.randn(2, 64, 16, 16)).shape == (2, 64, 16, 16))
check("3.3b 参数量 = 2*3*3*c*c + 4*c",
      lambda: n_params(BasicBlock(64)) == 2 * 3 * 3 * 64 * 64 + 4 * 64)
check("3.3c 梯度可以回传到输入（残差通路）",
      lambda: (lambda xx: (BasicBlock(64)(xx).sum().backward(), xx.grad is not None)[1])(
          torch.randn(2, 64, 8, 8, requires_grad=True)))


### 练习 3.4　SE 通道注意力

**要做什么**

实现 Squeeze-and-Excitation：

1. 全局平均池化得到每个通道的标量 `(N, C)`；
2. 两层全连接（先降维 `C -> C//16`，再升维 `C//16 -> C`），中间 ReLU，最后 sigmoid；
3. 把权重乘回原特征图的每个通道。

**提示**：`x.mean((2, 3))` 去掉空间维；权重广播时用 `[..., None, None]`。输出形状应与输入一致，且各通道被不同比例缩放。


In [ ]:
class SE(nn.Module):
    def __init__(self, c, r=16):
        super().__init__()
        # TODO

    def forward(self, x):
        # TODO
        return None


In [ ]:
_x = torch.randn(2, 64, 8, 8) + 3.0
_out = SE(64)(_x)
check("3.4a 输出形状不变", lambda: _out.shape == _x.shape)
check("3.4b 每个通道被同一个标量缩放",
      lambda: ((_out / _x).std(dim=(2, 3)) < 1e-4).all())
check("3.4c 通道权重不是全 1", lambda: ((_out / _x) - 1).abs().max() > 1e-3)


### 练习 3.5　感受野递推

**要做什么**

实现 `receptive_field(kernels, strides)`，用递推：

$$r_l=r_{l-1}+(k_l-1)\,j_{l-1},\qquad j_l=j_{l-1}\,s_l,\qquad r_0=1,\ j_0=1.$$

**提示**：三个 $3\times3$、步长 1 的卷积，感受野是 $7\times7$。


In [ ]:
def receptive_field(kernels, strides):
    return None   # TODO

rf_3 = None     # TODO：[3,3,3] / [1,1,1]
rf_5 = None     # TODO：[3]*5 / [1]*5
rf_mix = None   # TODO：[3,3,3] / [2,2,1]


In [ ]:
check("3.5a 三个 3x3 -> 7", lambda: rf_3 == 7)
check("3.5b 五个 3x3 -> 11", lambda: rf_5 == 11)
check("3.5c 带 stride 的混合栈 -> 15", lambda: rf_mix == 15)


In [ ]:
scoreboard()


**第三部分小结**：卷积输出尺寸与感受野是写网络时最常手算的两个量；深度可分离卷积把计算量降到约 1/8；残差块与 SE 模块是几乎所有现代 CNN 的积木。


## 第四部分　序列与注意力

### 练习 4.1　缩放点积注意力

**要做什么**

实现 `attention(q, k, v)`，输入形状均为 `(B, T, d)`：

$$\mathrm{Attention}(Q,K,V)=\mathrm{softmax}\left(\frac{QK^{\top}}{\sqrt{d}}\right)V$$

**提示**：`k.transpose(-2, -1)` 转置最后两维；缩放因子是 $1/\sqrt{d}$，不是 $1/d$。


In [ ]:
def attention(q, k, v):
    return None   # TODO

torch.manual_seed(0)
q = torch.randn(2, 6, 8)
k = torch.randn(2, 6, 8)
v = torch.randn(2, 6, 8)


In [ ]:
ref = torch.softmax(q @ k.transpose(-2, -1) / math.sqrt(q.size(-1)), dim=-1) @ v
check("4.1 与参考实现一致", lambda: close(attention(q, k, v), ref))


### 练习 4.2　因果掩码

**要做什么**

1. 构造 $6\times6$ 的布尔掩码 `mask`：**上三角（不含对角线）为 True**，表示这些位置要屏蔽；
2. 对随机分数矩阵 `scores` 做 `masked_fill(mask, -inf)` 后按行 softmax，得到 `attn`。

**提示**：`torch.triu(torch.ones(n, n, dtype=torch.bool), diagonal=1)`；softmax 的维度是 `-1`。


In [ ]:
n = 6
scores = torch.randn(n, n)

mask = None   # TODO
attn = None   # TODO


In [ ]:
check("4.2a 掩码正好是上三角",
      lambda: torch.equal(mask, torch.triu(torch.ones(n, n, dtype=torch.bool), diagonal=1)))
check("4.2b 每行权重和为 1",
      lambda: close(attn.sum(-1), torch.ones(n), 1e-5))
check("4.2c 未来位置权重为 0",
      lambda: attn[mask].abs().max() < 1e-6)
check("4.2d 第一个 token 只能看自己",
      lambda: close(attn[0], torch.tensor([1., 0., 0., 0., 0., 0.]), 1e-6))


### 练习 4.3　多头的拆分与合并

**要做什么**

1. `split_heads(x, n_heads)`：把 `(B, T, d)` 拆成 `(B, n_heads, T, d/n_heads)`；
2. `merge_heads(x)`：把 `(B, n_heads, T, d/n_heads)` 还原为 `(B, T, d)`。

**提示**：拆分后用 `transpose(1, 2)`；合并先 `transpose(1, 2)` 再 `reshape(B, T, d)`。两者必须严格互逆。


In [ ]:
def split_heads(x, n_heads):
    return None   # TODO

def merge_heads(x):
    return None   # TODO

torch.manual_seed(0)
x_heads = torch.randn(2, 5, 12)
heads = None        # TODO：split_heads(x_heads, 3)
restored = None     # TODO：merge_heads(heads)


In [ ]:
check("4.3a 拆分后的形状为 (2, 3, 5, 4)",
      lambda: heads.shape == (2, 3, 5, 4))
check("4.3b 合并后还原原张量",
      lambda: close(restored, x_heads))


### 练习 4.4　RoPE 的相对位置性质

**要做什么**

实现 `rope(x, positions, base=10000.0)`，输入 `x: (T, d)`（$d$ 为偶数），对每一对维度 $(2i, 2i+1)$ 按位置角 $\theta_i=\text{base}^{-2i/d}$ 做二维旋转。

**提示**：先算 `theta = base ** (-torch.arange(0, d, 2) / d)`，再令 `ang = positions[:, None] * theta[None, :]`；偶数维用 `x[..., 0::2]`，奇数维用 `x[..., 1::2]`，最后交错拼回。

**关键验证**：两个向量的内积只依赖位置差，不依赖绝对位置。


In [ ]:
def rope(x, positions, base=10000.0):
    return None   # TODO

torch.manual_seed(0)
xr = torch.randn(1, 16)
yr = torch.randn(1, 16)


In [ ]:
def _dot(a, b, pa, pb):
    return (rope(a, torch.tensor([pa])) * rope(b, torch.tensor([pb]))).sum()

check("4.4a 旋转保持范数",
      lambda: abs(rope(xr, torch.tensor([3])).norm().item() - xr.norm().item()) < 1e-5)
check("4.4b 内积只依赖相对位置（差为 4）",
      lambda: abs(_dot(xr, yr, 3, 7).item() - _dot(xr, yr, 103, 107).item()) < 1e-4)


### 练习 4.5　KV cache 显存估算

**要做什么**

计算下面配置下 KV cache 的字节数：

- 层数 `L=32`，batch `B=1`，序列长度 `T=8192`；
- KV 头数 `n_kv=8`（GQA），每个头维度 `d_head=128`；
- 每个元素 2 字节（fp16）。

**提示**：K 和 V 各存一份，公式为 `2 * L * B * T * n_kv * d_head * bytes`。


In [ ]:
L, B, T, n_kv, d_head, nbytes = 32, 1, 8192, 8, 128, 2
cache_bytes = None   # TODO


In [ ]:
check("4.5 KV cache 正好 1 GiB",
      lambda: cache_bytes == 2 ** 30)


### 练习 4.6　手写 LSTM 单步

**要做什么**

用 `nn.LSTM` 的权重手算 $t=0$ 的隐藏状态（初始 $h_0=c_0=0$），并与官方实现比较：

$$i,f,g,o=\text{chunk}(x_tW_{ih}^{\top}+hW_{hh}^{\top}+b_{ih}+b_{hh},4)$$

$$c=f\odot c_0+i\odot\tanh(g),\qquad h=o\odot\tanh(c)$$

**提示**：PyTorch 的门顺序是 **i, f, g, o**；`nn.LSTM(4, 5)` 的 `weight_ih_l0` 形状是 `(20, 4)`。


In [ ]:
torch.manual_seed(0)
lstm = nn.LSTM(4, 5)
x = torch.randn(1, 1, 4)
lstm_out, _ = lstm(x)

h_manual = None   # TODO


In [ ]:
check("4.6 手算 h 与 nn.LSTM 输出一致", lambda: close(lstm_out[0], h_manual, 1e-5))


In [ ]:
scoreboard()


**第四部分小结**：注意力的形状变换是 Transformer 实现的主要难点；因果掩码保证自回归；RoPE 把相对位置写进内积；GQA 的收益直接体现在 KV cache 显存上；LSTM 的门顺序稍不留神就会写错。


## 第五部分　生成模型与图神经网络

### 练习 5.1　VAE 的 KL 闭式解

**要做什么**

给定高斯后验 $q=\mathcal{N}(\mu,\sigma^2)$ 与标准正态先验，计算

$$\mathrm{KL}=-\frac12\sum_j\left(1+\log\sigma_j^2-\mu_j^2-\sigma_j^2\right).$$

**提示**：`logvar` 是 $\log\sigma^2$；逐项算完求和。


In [ ]:
mu = torch.tensor([1.0, -2.0])
logvar = torch.tensor([0.0, -1.0])

kl = None   # TODO


In [ ]:
check("5.1 KL 闭式解约为 2.6839", lambda: abs(kl.item() - 2.68394) < 1e-3)


### 练习 5.2　DDPM 的前向闭式采样

**要做什么**

给定 $\beta_t$ 从 $10^{-4}$ 线性到 $0.02$（1000 步），计算 $\alpha_t=1-\beta_t$、$\bar\alpha_t=\prod_{s\le t}\alpha_s$。

用 $x_0$、噪声 $\epsilon$ 和 $t=500$ 直接构造

$$x_t=\sqrt{\bar\alpha_t}\,x_0+\sqrt{1-\bar\alpha_t}\,\epsilon.$$

**提示**：`torch.cumprod` 计算累积乘积；下标 0 对应第 1 步。


In [ ]:
betas = torch.linspace(1e-4, 0.02, 1000)
alphas = 1 - betas
abar = torch.cumprod(alphas, dim=0)

torch.manual_seed(0)
x0 = torch.randn(1, 2)
eps = torch.randn(1, 2)
t = 500

xt = None   # TODO


In [ ]:
ref_xt = abar[t].sqrt() * x0 + (1 - abar[t]).sqrt() * eps
check("5.2 x_t 与闭式一致", lambda: close(xt, ref_xt))


### 练习 5.3　Top-p（nucleus）过滤

**要做什么**

实现 `top_p_filter(logits, p)`，返回布尔掩码（True 表示保留）：

1. 对 softmax 概率降序排列；
2. 取累积概率**首次达到或超过** $p$ 的最小前缀（含跨过 $p$ 的那个 token）；
3. 至少保留一个 token。

**提示**：`probs.sort(descending=True)` 返回值和下标；用 `cumsum` 判断前缀；最后按下标散射回原顺序。


In [ ]:
def top_p_filter(logits, p):
    return None   # TODO

tp_logits = torch.tensor([3.0, 2.0, 1.0, 0.0, -1.0])
mask_p09 = None   # TODO：p = 0.9


In [ ]:
probs = tp_logits.softmax(-1)
sorted_p, idx = probs.sort(descending=True)
cum = sorted_p.cumsum(0)
keep_sorted = (cum - sorted_p) < 0.9
ref_mask = torch.zeros_like(probs, dtype=torch.bool)
ref_mask[idx[keep_sorted]] = True

check("5.3a 掩码与参考实现一致", lambda: torch.equal(mask_p09, ref_mask))
check("5.3b 掩码是布尔张量且保留概率质量 >= 0.9",
      lambda: mask_p09.dtype == torch.bool and probs[mask_p09].sum() >= 0.9 - 1e-6)
check("5.3c 至少保留一个 token", lambda: mask_p09.sum() >= 1)


### 练习 5.4　手写一层 GCN

**要做什么**

给定邻接矩阵 $A$、特征 $H$、权重 $W$，实现

$$H'=\sigma\left(\tilde D^{-\frac12}\tilde A\tilde D^{-\frac12}HW\right),\qquad \tilde A=A+I.$$

**提示**：`torch.diag` 构造度矩阵；$D^{-1/2}$ 用 `D.pow(-0.5)`；激活用 ReLU。这个 $3\times3$ 图里所有节点都有自环，度矩阵不会出现 0。


In [ ]:
A = torch.tensor([[0., 1., 1.],
                  [1., 0., 1.],
                  [1., 1., 0.]])
H = torch.randn(3, 4)
W = torch.randn(4, 5)

A_norm = None   # TODO
gcn_out = None  # TODO


In [ ]:
A_tilde = A + torch.eye(3)
D = A_tilde.sum(-1)
D_inv_sqrt = torch.diag(D.pow(-0.5))
ref_A = D_inv_sqrt @ A_tilde @ D_inv_sqrt
ref_out = F.relu(ref_A @ H @ W)

check("5.4a 归一化邻接矩阵正确", lambda: close(A_norm, ref_A))
check("5.4b 输出正确", lambda: close(gcn_out, ref_out))
check("5.4c 归一化邻接矩阵对称", lambda: close(A_norm, A_norm.T))


In [ ]:
scoreboard()


**第五部分小结**：VAE 的 KL 有闭式，扩散的前向过程也是一步闭式采样；top-p 是自适应的截断采样；GCN 的核心就是「对称归一化邻接矩阵 + 线性变换 + 激活」。


---

## 第六部分　自测选择题

把答案填成 A/B/C/D，运行下一格自测。答案也附在本节末尾。

**6.1** 深层 sigmoid 网络梯度消失的根本原因是：
A. 学习率过大　B. 每层导数上界 0.25，连乘后指数衰减　C. 参数太多　D. 数据太少

**6.2** 残差连接主要解决：
A. 过拟合　B. 退化问题与梯度流　C. 类别不均衡　D. 显存不足

**6.3** Transformer 中注意力分数除以 $\sqrt{d_k}$ 的作用是：
A. 加速计算　B. 防止 softmax 饱和、稳定梯度　C. 正则化　D. 注入位置信息

**6.4** 下列哪个激活函数是零中心的：
A. ReLU　B. Sigmoid　C. Tanh　D. 以上都不是

**6.5** `CrossEntropyLoss` 的 input 参数应该传：
A. 概率　B. logits　C. one-hot 标签　D. 类别索引

**6.6** 极端类别不均衡下，最常用来聚焦难样本的损失是：
A. MSE　B. Focal Loss　C. MAE　D. KL 散度

**6.7** RoPE 的核心性质是：
A. 绝对位置编码　B. 内积只依赖相对位置　C. 降维　D. 稀疏化

**6.8** GQA 主要节省：
A. 参数量　B. KV cache 显存　C. 训练数据　D. 词表大小

**6.9** VAE 中重参数化技巧的目的是：
A. 让采样可导　B. 加速训练　C. 防止过拟合　D. 提高生成分辨率

**6.10** Diffusion 的训练目标通常是：
A. 分类交叉熵　B. 噪声回归 MSE　C. 对抗损失　D. 对比损失

**6.11** BatchNorm 在推理时使用：
A. 当前 batch 统计量　B. 训练时维护的滑动平均统计量　C. 不使用统计量　D. 随机统计量

**6.12** MoE 的负载均衡损失用于：
A. 防止路由崩塌　B. 加速推理　C. 数据增强　D. 量化


In [ ]:
q1 = None    # 6.1
q2 = None    # 6.2
q3 = None    # 6.3
q4 = None    # 6.4
q5 = None    # 6.5
q6 = None    # 6.6
q7 = None    # 6.7
q8 = None    # 6.8
q9 = None    # 6.9
q10 = None   # 6.10
q11 = None   # 6.11
q12 = None   # 6.12


In [ ]:
quiz("6.1 梯度消失的原因", q1, "B")
quiz("6.2 残差解决什么", q2, "B")
quiz("6.3 缩放因子的作用", q3, "B")
quiz("6.4 零中心激活", q4, "C")
quiz("6.5 CE 的输入", q5, "B")
quiz("6.6 不均衡损失", q6, "B")
quiz("6.7 RoPE 的性质", q7, "B")
quiz("6.8 GQA 省什么", q8, "B")
quiz("6.9 重参数化", q9, "A")
quiz("6.10 扩散训练目标", q10, "B")
quiz("6.11 BN 推理", q11, "B")
quiz("6.12 MoE 辅助损失", q12, "A")

scoreboard()


---

## 参考答案

### 练习 1.1–1.6

```python
# 1.1
y_two = l2(l1(x))
W = l2.weight @ l1.weight
b = l2.weight @ l1.bias + l2.bias
y_one = x @ W.T + b

# 1.2
def sigmoid_t(x):
    return 1 / (1 + torch.exp(-x))
max_deriv = 0.25

# 1.3
def gelu_exact(x):
    return 0.5 * x * (1 + torch.erf(x / math.sqrt(2)))

def gelu_tanh(x):
    return 0.5 * x * (1 + torch.tanh(math.sqrt(2 / math.pi) * (x + 0.044715 * x ** 3)))

# 1.4
def silu_t(x):
    return x * torch.sigmoid(x)
swiglu_params = 3 * d * d_ff

# 1.5
def std_ratio(init_fn, depth=30, width=256):
    torch.manual_seed(0)
    h = torch.randn(64, width)
    stds = []
    for _ in range(depth):
        lin = nn.Linear(width, width)
        init_fn(lin)
        h = F.relu(lin(h))
        stds.append(h.std().item())
    return stds[-1] / stds[0]

# 1.6
def grad_norm(depth, act, residual=False):
    torch.manual_seed(0)
    layers = []
    for _ in range(depth):
        lin = nn.Linear(64, 64)
        nn.init.xavier_uniform_(lin.weight)
        layers.append(lin)
    x = torch.randn(32, 64, requires_grad=True)
    h = x
    for lin in layers:
        z = act(lin(h))
        h = h + z if residual else z
    h.pow(2).mean().backward()
    return x.grad.norm().item()
```

### 练习 2.1–2.6

```python
# 2.1
def ce_from_logits(logits, target):
    m = logits.max(dim=-1, keepdim=True).values
    logsumexp = (logits - m).exp().sum(-1, keepdim=True).log() + m
    log_probs = logits - logsumexp
    return -log_probs.gather(1, target[:, None]).squeeze(1).mean()

# 2.2
soft_target = torch.full((K,), eps / K)
soft_target[y] += 1 - eps
min_ce = -(soft_target * soft_target.log()).sum()

# 2.3
def focal_loss(pt, gamma=2.0, alpha=1.0):
    return -alpha * (1 - pt) ** gamma * pt.log()
fl_g0 = focal_loss(pt, gamma=0.0)
fl_g2 = focal_loss(pt, gamma=2.0)

# 2.4
inter = (pred * target).sum()
dice = 2 * inter / (pred.sum() + target.sum())
iou = inter / ((pred + target) > 0).sum()

# 2.5
def _entropy(tau):
    p = (sim / tau).softmax(-1)
    return -(p * p.clamp_min(1e-12).log()).sum(-1).mean()
ent_tau_1 = _entropy(1.0)
ent_tau_small = _entropy(0.07)

# 2.6
loss_plain = F.cross_entropy(logits, target)
loss_weighted = F.cross_entropy(logits, target, weight=w)
```

### 练习 3.1–3.5

```python
# 3.1
def conv_out(size, k, s=1, p=0, d=1):
    return (size + 2 * p - d * (k - 1) - 1) // s + 1

out_a, out_b, out_c, out_d = conv_out(32, 3, p=1), conv_out(224, 7, 2, 3), conv_out(32, 3, 2, 1), conv_out(32, 3, 1, 1, 2)

# 3.2
std_params = kk * kk * Cin * Cout
dw_params = kk * kk * Cin + Cin * Cout
ratio = dw_params / std_params

# 3.3
class BasicBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.conv1 = nn.Conv2d(c, c, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(c)
        self.conv2 = nn.Conv2d(c, c, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(c)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + x)

# 3.4
class SE(nn.Module):
    def __init__(self, c, r=16):
        super().__init__()
        self.fc1 = nn.Linear(c, c // r)
        self.fc2 = nn.Linear(c // r, c)

    def forward(self, x):
        s = x.mean(dim=(2, 3))
        s = torch.sigmoid(self.fc2(F.relu(self.fc1(s))))
        return x * s[..., None, None]

# 3.5
def receptive_field(kernels, strides):
    r, j = 1, 1
    for k, s in zip(kernels, strides):
        r = r + (k - 1) * j
        j = j * s
    return r
```

### 练习 4.1–4.6

```python
# 4.1
def attention(q, k, v):
    return torch.softmax(q @ k.transpose(-2, -1) / math.sqrt(q.size(-1)), dim=-1) @ v

# 4.2
mask = torch.triu(torch.ones(n, n, dtype=torch.bool), diagonal=1)
attn = scores.masked_fill(mask, float("-inf")).softmax(-1)

# 4.3
def split_heads(x, n_heads):
    B, T, d = x.shape
    return x.view(B, T, n_heads, d // n_heads).transpose(1, 2)

def merge_heads(x):
    B, h, T, dh = x.shape
    return x.transpose(1, 2).reshape(B, T, h * dh)

heads = split_heads(x_heads, 3)
restored = merge_heads(heads)

# 4.4
def rope(x, positions, base=10000.0):
    T, d = x.shape
    theta = base ** (-torch.arange(0, d, 2).float() / d)
    ang = positions.float()[:, None] * theta[None, :]
    x1, x2 = x[:, 0::2], x[:, 1::2]
    out = torch.stack(
        [x1 * ang.cos() - x2 * ang.sin(), x1 * ang.sin() + x2 * ang.cos()], dim=-1
    )
    return out.flatten(-2)

# 4.5
cache_bytes = 2 * L * B * T * n_kv * d_head * nbytes

# 4.6
h0 = torch.zeros(1, 5)
c0 = torch.zeros(1, 5)
gates = x[0] @ lstm.weight_ih_l0.T + h0 @ lstm.weight_hh_l0.T + lstm.bias_ih_l0 + lstm.bias_hh_l0
i, f, g, o = gates.chunk(4, dim=1)
c = torch.sigmoid(f) * c0 + torch.sigmoid(i) * torch.tanh(g)
h_manual = torch.sigmoid(o) * torch.tanh(c)
```

### 练习 5.1–5.4

```python
# 5.1
kl = (-0.5 * (1 + logvar - mu ** 2 - logvar.exp())).sum()

# 5.2
xt = abar[t].sqrt() * x0 + (1 - abar[t]).sqrt() * eps

# 5.3
def top_p_filter(logits, p):
    probs = logits.softmax(-1)
    sorted_p, idx = probs.sort(descending=True)
    cum = sorted_p.cumsum(0)
    keep = (cum - sorted_p) < p
    keep[0] = True
    mask = torch.zeros_like(probs, dtype=torch.bool)
    mask[idx[keep]] = True
    return mask

mask_p09 = top_p_filter(tp_logits, 0.9)

# 5.4
A_tilde = A + torch.eye(3)
D_inv_sqrt = torch.diag(A_tilde.sum(-1).pow(-0.5))
A_norm = D_inv_sqrt @ A_tilde @ D_inv_sqrt
gcn_out = F.relu(A_norm @ H @ W)
```

### 自测选择题答案

| 题号 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 | 11 | 12 |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 答案 | B | B | B | C | B | B | B | B | A | B | B | A |

---

## 结语

如果 27 个练习里能独立完成 20 个以上，说明你已经把教材里「可验证的部分」真正掌握了。剩下的工作是把这些碎片连成系统：

- 能看着公式说出「它在解决什么问题」；
- 能看着一个网络结构说出「它为什么这样设计」；
- 能在新任务上做出结构、激活、损失、优化器的合理选择。

回到教材第 12 章的选型表，找一个你感兴趣的真实任务，从零搭一个基线——那才是最后一题。
